# Phantom_VisionHub Native Classifier Colab

This notebook trains the native torchvision classifier family shipped in this repository's `visionhub` package.

Expected dataset layout:

```text
dataset_root/
  train/
    class_a/
      image1.jpg
  val/
    class_a/
      image2.jpg
```


In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not running inside Colab; skipping Google Drive mount.')


In [ ]:
GDRIVE_REPO_PATH = '/content/drive/MyDrive/Phantom_VisionHub'
REPO_LOCAL_PATH = '/content/Phantom_VisionHub'

DATA_ROOT = '/content/classifier_dataset'
OUTPUT_ROOT = '/content/classifier_output/efficientnet_v2_s_run1'

ARCHITECTURE = 'efficientnet_v2_s'
EPOCHS = 50
BATCH_SIZE = 16
IMAGE_SIZE = 224
PATIENCE = 50
AUGMENTATION_PRESET = 'phantom'  # none | reference | phantom
DEVICE = 'auto'
TOPK = 5
REPORT_FILE = 'classifier_report.txt'


## Copy And Install

Installs this repository in editable mode with ONNX export extras.

In [ ]:
import os
import shutil
import subprocess
import sys

if os.path.isdir(REPO_LOCAL_PATH):
    shutil.rmtree(REPO_LOCAL_PATH)
if not os.path.isdir(GDRIVE_REPO_PATH):
    raise FileNotFoundError(f'Repository folder not found on Google Drive: {GDRIVE_REPO_PATH}')
shutil.copytree(GDRIVE_REPO_PATH, REPO_LOCAL_PATH)
print(f'Repository copied to {REPO_LOCAL_PATH}')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-U', 'pip'], check=True)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-e', f'{REPO_LOCAL_PATH}[export,notebook]'
], check=True)


## Train Classifier

Runs `visionhub-train-classifier` and streams logs live.

In [ ]:
import os
import shlex
import shutil
import subprocess
import sys

train_cli = shutil.which('visionhub-train-classifier') or os.path.join(os.path.dirname(sys.executable), 'visionhub-train-classifier')
cmd = [
    train_cli,
    '--source_root', DATA_ROOT,
    '--output_root', OUTPUT_ROOT,
    '--architecture', ARCHITECTURE,
    '--epochs', str(EPOCHS),
    '--batch_size', str(BATCH_SIZE),
    '--image_size', str(IMAGE_SIZE),
    '--patience', str(PATIENCE),
    '--optimizer', 'auto',
    '--augmentation_preset', AUGMENTATION_PRESET,
    '--use-pretrained-weights',
    '--device', DEVICE,
]

print('Command:')
print(' '.join(shlex.quote(part) for part in cmd))

process = subprocess.Popen(
    cmd,
    cwd=REPO_LOCAL_PATH,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in process.stdout:
    print(line, end='')

process.wait()
if process.returncode != 0:
    raise RuntimeError(f'Training failed with exit code {process.returncode}')


## Inference And Export

Runs checkpoint inference on the validation split and exports the best checkpoint to ONNX.

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

checkpoint_path = os.path.join(OUTPUT_ROOT, 'training_run', 'weights', 'least_confused.pt')
infer_output = os.path.join(OUTPUT_ROOT, 'inference_predictions')
infer_source = os.path.join(DATA_ROOT, 'val')

infer_cli = shutil.which('visionhub-infer-classifier') or os.path.join(os.path.dirname(sys.executable), 'visionhub-infer-classifier')
subprocess.run([
    infer_cli,
    '--checkpoint', checkpoint_path,
    '--source', infer_source,
    '--output', infer_output,
    '--topk', str(TOPK),
    '--report-file', REPORT_FILE,
], cwd=REPO_LOCAL_PATH, check=True)

export_cli = shutil.which('visionhub-export-classifier-onnx') or os.path.join(os.path.dirname(sys.executable), 'visionhub-export-classifier-onnx')
subprocess.run([
    export_cli,
    '--checkpoint', checkpoint_path,
    '--check',
    '--simplify',
], cwd=REPO_LOCAL_PATH, check=True)

print(f'Inference outputs: {infer_output}')
print('ONNX exports are written under:')
print(os.path.join(REPO_LOCAL_PATH, 'onnx_engines'))
print('The generic TensorRT helpers in this repository can consume the exported classifier ONNX model as well.')
